In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

In [3]:
# formulas usadas:
# d_a = Medida de las 20 hojas de acetato en conjunto
# d_p = Medida de las 20 hojas de papel en conjunto
# A = Área de cada placa circular
# K_a = C*d / (A * *eps0)  Constante dieléctrica para la hoja de acetato (Ecuación 1 del pdf)
# eps0 = 8.85e-12 permitividad en el vacío


archivo = pd.read_excel('LAB5.xlsx', sheet_name='Sheet1')
df = pd.DataFrame(archivo)

df

,Unnamed: 0,Capacitancia [pF],Acetato,Papel,Unnamed: 4,Grosor 20 acetatos [mm],Grosor 20 papeles,palmer incertidumbre,diametro,regla incertidumbre,pf incert
0,NaN,141.52,20,0,NaN,2.0,1.77,0.01,11.9,0.05,0.01
1,NaN,142.34,19,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,142.77,18,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,143.32,17,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,145.48,16,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,147.85,15,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,149.62,14,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,150.97,13,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,151.34,12,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,153.64,11,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
C_pF = df["Capacitancia [pF]"] # Capacitancia en pico Faradios
Hojas = df["Acetato"]

Cap_F = C_pF * 1e-12

d_a = (2 * 10**(-3)) / 20 #Se divide para 20 porque son 20 láminas
d_p = (1.77 * 10**(-3)) / 20 #Se divide para 20 porque son 20 láminas
A = (11.9 * 10**(-2) / 2) ** 2 * np.pi
eps0 = 8.85e-12 
K_a = Cap_F[0] * d_a * 20/ (A * eps0) 

Delta_C = 1e-14 #incertidumbre capacitancia en faradios
Delta_A = 9.345 * 1e-5 #incertidumbre area en metros
Delta_d = 1e-5 #incertidumbre grosor de las 20 laminas de acetato juntas en metros

#Cap_F[0] es la capacitancia para las 20 laminas de acetato juntas.

Delta_Ka = np.sqrt((d_a * eps0 * Delta_C / A)**2 + (Cap_F[0] * eps0 * Delta_d / A)**2 + (Cap_F[0] * d_a * eps0 / A**2)**2)

Kp = []

for i in range(len(Cap_F)):
    num_acetato = 20 - i
    
    #Espesor en cada paso para acetato y papel
    dp = i * d_p
    da = num_acetato * d_a

    Kpi = (dp * K_a * Cap_F[i]) / ((K_a * eps0 * A) - (Cap_F[i] * da))

    Kp.append(Kpi)

K_p = np.array(Kp, dtype = float)
Promedio = K_p[1:-1].mean() #Eliminamos la primera y la última toma porque son de control

Delta_Kp = []

for i in range(len(Cap_F)):
    num_acetato = 20 - i
    
    #Espesor en cada paso para acetato y papel
    dp = i * d_p
    da = num_acetato * d_a

    Kpi = np.sqrt((K_a * Cap_F[i] * Delta_d / (K_a * eps0 * A - Cap_F[i] * da))**2 + (dp * K_a**2 * eps0 * A * Delta_C / (K_a * eps0 * A - Cap_F[i] * da)**2)**2 +
                  dp * K_a * Cap_F[i]**2 * Delta_d / (K_a * eps0 * A - Cap_F[i] * da)**2)

    Delta_Kp.append(Kpi)

# Cálculo de la permitividad del papel.
ep = K_p * eps0
Promedio2 = ep[1:-1].mean()

StdK = np.std(K_p)
Stde = np.std(ep)

Error1 = StdK / 20
Error2 = Stde / 20

Porcentual = abs(3.7 - Promedio) * 100 / 3.7

print(f"Kp obtenido: {Promedio:.3f}")
print(f"Ka obtenido: {K_a:.3f}")
print(f"Incertidumbre de Ka: {Delta_Ka:.3f}")
print(f"ep obtenido: {Promedio2:.3e}")
print(f"Incertidumbre de Kp: {Error1:.3f}")
print(f"Incertidumbre de ep: {Error2:.3e}")
print(f"Error porcentual: {Porcentual:.3f} %")
#print(f"error promedio: {error_promedio:.2e}")


Kp obtenido: 3.126
Ka obtenido: 2.876
Incertidumbre de Ka: 0.000
ep obtenido: 2.766e-11
Incertidumbre de Kp: 0.034
Incertidumbre de ep: 3.042e-13
Error porcentual: 15.522 %


In [32]:
for i in range(len(ep)-1):
    
    print(f" {ep[i+1]:.3e} &", end=" ")
    
    if (i+1) % 5 == 0:
        print()

 2.545e-11 &  2.468e-11 &  2.458e-11 &  2.607e-11 &  2.718e-11 & 
 2.748e-11 &  2.743e-11 &  2.688e-11 &  2.731e-11 &  2.797e-11 & 
 2.954e-11 &  2.929e-11 &  2.895e-11 &  2.888e-11 &  2.886e-11 & 
 2.870e-11 &  2.882e-11 &  2.869e-11 &  2.882e-11 &  2.917e-11 & 


In [33]:
for i in range(len(Kp)-1):
    epd = ep[i+1] * eps0
    print(f" $\pm$ {epd:.3e} &", end=" ")
    
    if (i+1) % 5 == 0:
        print()



 $\pm$ 2.253e-22 &  $\pm$ 2.184e-22 &  $\pm$ 2.175e-22 &  $\pm$ 2.307e-22 &  $\pm$ 2.405e-22 & 
 $\pm$ 2.432e-22 &  $\pm$ 2.427e-22 &  $\pm$ 2.379e-22 &  $\pm$ 2.417e-22 &  $\pm$ 2.475e-22 & 
 $\pm$ 2.614e-22 &  $\pm$ 2.592e-22 &  $\pm$ 2.562e-22 &  $\pm$ 2.556e-22 &  $\pm$ 2.554e-22 & 
 $\pm$ 2.540e-22 &  $\pm$ 2.550e-22 &  $\pm$ 2.539e-22 &  $\pm$ 2.551e-22 &  $\pm$ 2.581e-22 & 


<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\andy\AppData\Local\Temp\ipykernel_22376\3387567354.py:3: SyntaxWarning: invalid escape sequence '\p'
  print(f" $\pm$ {epd:.3e} &", end=" ")


In [18]:
for i in range(len(Kp)-1):
    
    print(f" {Kp[i+1]:.3f} &", end=" ")
    
    if (i+1) % 5 == 0:
        print()

 2.876 &  2.789 &  2.777 &  2.946 &  3.071 & 
 3.105 &  3.099 &  3.038 &  3.086 &  3.160 & 
 3.338 &  3.310 &  3.271 &  3.264 &  3.261 & 
 3.243 &  3.256 &  3.242 &  3.257 &  3.296 & 


In [16]:

for i in range(len(C_pF)):
    
    print(f" {C_pF[i]:.3f} &", end=" ")
    
    if (i+1) % 5 == 0:
        print()

 141.520 &  142.340 &  142.770 &  143.320 &  145.480 & 
 147.850 &  149.620 &  150.970 &  151.340 &  153.640 & 
 156.780 &  162.790 &  164.300 &  165.380 &  167.320 & 
 169.420 &  170.970 &  173.790 &  175.460 &  178.620 & 
 183.290 & 